# Loadery dokumentów i podział na chunki – zadania

## Loadery plików .txt, .pdf i .docx

1. Wczytaj tekst "Pana Tadeusza" z pliku `documents/pan-tadeusz.txt`, korzystając z klasy `TextLoader` albo `KreuzbergLoader`.

Sprawdź liczbę zwróconych dokumentów, metadane oraz długość i początek treści dokumentu.

---
(czas: 3 min.)

In [ ]:
# ...

2. Wczytaj tekst "Pana Tadeusza" z pliku `documents/pan-tadeusz.pdf`, korzystając z klasy `PyMuPDFLoader` albo `KreuzbergLoader` (w przypadku `KreuzbergLoader` użyj konfiguracji `PageConfig(extract_pages=True)`, aby podzielić plik na strony).

Sprawdź liczbę zwróconych dokumentów, metadane oraz treść wybranej strony.

---
(czas: 4 min.)

In [ ]:
# ...

## Własne loadery i loadery API

3. API Wolnych Lektur zwraca JSON z metadanymi książki, w tym adres pliku tekstowego w polu `txt`. Napisz własny loader `WolneLekturyLoader(BaseLoader)`, który pobierze z niego pełną treść kilku książek – każdą jako osobny `Document`. Nie jest wymagany klucz API.

**Kroki:**
1. Konstruktor przyjmuje listę `slugs` (identyfikatorów książek, np. `"pan-tadeusz"`).
2. W metodzie `lazy_load()` dla każdego `slug`-a:
   - pobierz metadane z `https://wolnelektury.pl/api/books/<slug>/` (użyj `requests.get()` z `timeout=30` oraz `raise_for_status()`),
   - pobierz treść spod adresu z pola `txt` i zwróć `Document`, którego `page_content` to pełny tekst książki.
3. Metadane dokumentu: `title`, `author` (autorzy połączeni przecinkiem), `language`, `source` (pole `url`), `text_url` (pole `txt`) oraz `slug`.
4. Przetestuj loader na książkach: `pan-tadeusz`, `balladyna` oraz `w-pustyni-i-w-puszczy`.

Sprawdź liczbę zwróconych dokumentów, metadane każdego z nich oraz długość i początek treści wybranej książki.

**Oczekiwany wynik:** lista `documents`, w której każdy element wygląda podobnie do:

```python
Document(
    page_content="Adam Mickiewicz\n\nPan Tadeusz, czyli ostatni zajazd na Litwie\n\n...",
    metadata={
        "title": "Pan Tadeusz, czyli ostatni zajazd na Litwie",
        "author": "Adam Mickiewicz",
        "language": "pol",
        "source": "https://wolnelektury.pl/katalog/lektura/pan-tadeusz/",
        "text_url": "https://wolnelektury.pl/media/book/txt/pan-tadeusz.txt",
        "slug": "pan-tadeusz",
    },
)
```

---
(czas: 12 min.)

## Rodzaje splitterów, dobór parametrów i własne metadane chunków

4. Wczytaj tekst pierwszej księgi "Pana Tadeusza" z pliku `documents/pan-tadeusz_księga_1.txt`, a następnie podziel go na chunki używając `CharacterTextSplitter` z separatorem `"\n\n"` oraz metody `split_text()`. Potestuj różne wartości `chunk_size` oraz `chunk_overlap`.

Sprawdź liczbę chunków oraz długość każdego z nich. Przyjrzyj się kilku kolejnym chunkom i ich overlapom.

---
(czas: 6 min.)

In [ ]:
# ...

---
---
---

5. W folderze `documents/pan-tadeusz/` znajduje się cały "Pan Tadeusz" podzielony na 13 plików PDF – po jednym na każdą księgę (`01_księga_pierwsza.pdf` … `12_księga_dwunasta.pdf`) oraz `13_epilog.pdf`. Przygotuj z nich listę chunków gotowych do zapisania w bazie wektorowej.

**Kroki:**
1. Znajdź wszystkie pliki `.pdf` w folderze `documents/pan-tadeusz/` i posortuj je po nazwie.
2. Wczytaj każdy plik za pomocą `PyMuPDFLoader` (każda strona PDF-a to osobny `Document`).
3. Podziel wszystkie wczytane dokumenty na chunki za pomocą `RecursiveCharacterTextSplitter` z separatorami `"\n\n"`, `"\n"`, `". "`. Użyj metody `split_documents()`, dzięki czemu każdy chunk zachowa metadane strony, z której pochodzi. Dobierz `chunk_size` oraz `chunk_overlap`.
4. Do metadanych każdego chunka dodaj:
   - `book_number` – numer księgi jako `int`, wzięty z początku nazwy pliku (np. `9` dla `09_księga_dziewiąta.pdf`); dla epilogu (`13_epilog.pdf`) wartość `None`,
   - `chunk_length` – długość treści chunka (liczba znaków `page_content`).
5. Pozostaw w metadanych tylko pola: `source`, `page`, `title`, `book_number`, `chunk_length` (pozostałe metadane z `PyMuPDFLoader`, np. `producer` czy `creationdate`, usuń).

**Oczekiwany wynik:** lista `chunks` obiektów klasy `Document`, w której każdy element wygląda podobnie do:

```python
Document(
    page_content="Księga pierwsza – Gospodarstwo\nPowrót panicza — Spotkanie się pierwsze w pokoiku, ...",
    metadata={
        "source": "documents/pan-tadeusz/01_księga_pierwsza.pdf",
        "page": 0,
        "title": "Księga pierwsza – Gospodarstwo",
        "book_number": 1,
        "chunk_length": 964
    }
)
```

Sprawdź łączną liczbę chunków, liczbę chunków w każdej księdze oraz metadane chunków z epilogu.

---
(czas: 15 min.)

In [ ]:
# ...

---
---
---

6. W folderze `documents/pan-tadeusz/` znajdują się również pliki `.txt` z treścią poszczególnych ksiąg. Każdy z nich zaczyna się od nagłówka, np. `# Księga pierwsza – Gospodarstwo`. Tym razem wczytaj je **bez loadera**, jako zwykły tekst, i podziel na chunki ze względu na liczbę tokenów.

**Kroki:**
1. Wczytaj treść wszystkich plików `.txt` z folderu `documents/pan-tadeusz/` (posortowanych po nazwie) do listy stringów `texts`
2. Przygotuj listę `metadatas` – po jednym słowniku dla każdego tekstu – z polami:
   - `source` – ścieżka do pliku,
   - `title` – tytuł wzięty z pierwszej linii pliku, bez znaków `# ` na początku (np. `"Księga pierwsza – Gospodarstwo"`)
3. Podziel teksty na chunki za pomocą `TokenTextSplitter`, który liczy tokeny tak jak model `gpt-4o-mini` (zamiast nazwy kodowania podaj nazwę modelu w parametrze `model_name`). Użyj metody `create_documents(texts, metadatas=metadatas)`. Dobierz `chunk_size` oraz `chunk_overlap`
4. Do metadanych każdego chunka dodaj `token_count` – liczbę tokenów w treści chunka, policzoną za pomocą `tiktoken` dla modelu `gpt-4o-mini` (`tiktoken.encoding_for_model()`)

**Oczekiwany wynik:** lista `chunks` obiektów klasy `Document`, w której każdy element wygląda podobnie do:

```python
Document(
    page_content="# Księga pierwsza – Gospodarstwo\n\nPowrót panicza — Spotkanie się pierwsze w pokoiku, ...",
    metadata={
        "source": "documents/pan-tadeusz/01_księga_pierwsza.txt",
        "title": "Księga pierwsza – Gospodarstwo",
        "token_count": 400
    }
)
```

Sprawdź łączną liczbę chunków i porównaj ją z wynikiem zadania 4. Sprawdź też, które chunki mają mniej tokenów niż ustawiony `chunk_size` i dlaczego.

---
(czas: 10 min.)

In [ ]:
# ...